# Day 24 — Lecture 4: Overfitting

> Source: Apna College Prime AI/ML • Day 24 Lecture 4 • Video Duration: `07:06`
> Core Objective: Overfitting ko samajhna — jab model training data (outliers samet) ko memorize kar leta hai par test data par fail hota hai.
> Key Parameters: `PolynomialFeatures(degree=10)`, `LinearRegression()`, `r2_score`
> Dataset Used: Synthetic Curve Data (Non-linear Pattern + Outlier)
> Core Concept: Overfit model bohot complex hota hai — isme **Low Bias** (training error ~ 0) aur **High Variance** (naye data par bohot sensitive) hota hai

## 🛠️ Step 0: Setup & Non-Linear Data Create Karo

Chalo ek curved pattern par synthetic training aur test dataset banate hain jisme ek outlier point bhi shamil ho.

In [ ]:
# 1. Libraries import karo
import numpy as np
import matplotlib.pyplot as plt
from sklearn.preprocessing import PolynomialFeatures
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

# 2. Training data banao (quadratic curve + slight noise + ek outlier)
np.random.seed(42)
X_train = np.linspace(-3, 3, 12).reshape(-1, 1)
y_train = X_train.ravel() ** 2 + np.random.normal(0, 0.4, 12)
y_train[5] += 4.5  # Outlier point jisse overfit model pakad lega

# 3. Unseen test data create karo
X_test = np.linspace(-2.8, 2.8, 20).reshape(-1, 1)
y_test = X_test.ravel() ** 2
print("Train points:", len(X_train), "| Test points:", len(X_test))

## 📍 Step 1: Normal Fit vs Complex Overfit Curve

Lecture me samjhaya gaya: `model.fit()` ka kaam pattern seekhna hai, data ko ratna (memorize karna) nahi.
- **Normal Fit (Degree 2):** Smooth curve banata hai jo asli trend capture karta hai.
- **Overfit Model (Degree 10):** Har single training point aur outlier se guzar kar training error ($J(\theta)$) ko almost `0` kar deta hai.

In [ ]:
# 1. Helper function banao jo kisi bhi polynomial degree par model train kare
def fit_poly(deg, X_tr, y_tr):
    poly = PolynomialFeatures(degree=deg)
    model = LinearRegression()
    model.fit(poly.fit_transform(X_tr), y_tr)
    return poly, model

# 2. Optimal (Degree 2) aur Overfit (Degree 10) models fit karo
poly_opt, opt_model = fit_poly(2, X_train, y_train)
poly_ovr, ovr_model = fit_poly(10, X_train, y_train)
print("Dono models train ho gaye!")

## ⚠️ Step 2: Training High vs Testing Low Accuracy Gap

Dekho yahi overfitting ki sabse badi pehchan hai — jaise lecture example me Training `95%` par Test sirf `65%–75%` reh jati hai.
- **Rule:** Sirf training error `0` dekh kar kabhi khush nahi hona chahiye; asli test hamesha unseen data par hota hai.

In [ ]:
# 1. Optimal model ka Train vs Test R2 check karo
opt_tr = r2_score(y_train, opt_model.predict(poly_opt.transform(X_train)))
opt_te = r2_score(y_test, opt_model.predict(poly_opt.transform(X_test)))

# 2. Overfit model ka Train vs Test R2 check karo
ovr_tr = r2_score(y_train, ovr_model.predict(poly_ovr.transform(X_train)))
ovr_te = r2_score(y_test, ovr_model.predict(poly_ovr.transform(X_test)))

# 3. Accuracy drop compare karo
print(f"Optimal (Deg 2) -> Train R2: {opt_tr:.3f} | Test R2: {opt_te:.3f}")
print(f"Overfit (Deg 10)-> Train R2: {ovr_tr:.3f} | Test R2: {ovr_te:.3f}")

## 🎨 Step 3: Overfit Curve Visualize Karo

Chalo graph par dekhte hain ki overfit curve kaise har point ko chhoone ke chakkar me ajeeb zig-zag karta hai.

In [ ]:
# 1. Smooth plotting grid banao
X_grid = np.linspace(-3, 3, 200).reshape(-1, 1)
y_opt_grid = opt_model.predict(poly_opt.transform(X_grid))
y_ovr_grid = ovr_model.predict(poly_ovr.transform(X_grid))

# 2. Training points ke saath dono curves plot karo
plt.scatter(X_train, y_train, color="black", label="Train Data")
plt.plot(X_grid, y_opt_grid, color="green", label="Normal Fit (Deg 2)")
plt.plot(X_grid, y_ovr_grid, color="red", linestyle="--", label="Overfit (Deg 10)")
plt.ylim(-2, 14)
plt.legend()
plt.title("Optimal Fit vs Overfitting")
plt.show()

## 🔬 Step 4: Low Bias vs High Variance Verify Karo

Yaad rakhna: Yahan **Bias** ka matlab intercept $\theta_0$ nahi, balki **error from wrong assumptions** hai.
- **Low Bias:** Overfit model wrong assumptions nahi karta (training data perfectly memorize karta hai).
- **High Variance:** Data me sirf 2 naye points add karne par complex curve bohot zyada badal (vary) jata hai!

In [ ]:
# 1. Training data me sirf 2 naye points add karo
X_mod = np.vstack([X_train, [[-0.2], [0.2]]])
y_mod = np.append(y_train, [0.1, 2.5])

# 2. Dono models ko updated data par retrain karo
_, opt_mod = fit_poly(2, X_mod, y_mod)
_, ovr_mod = fit_poly(10, X_mod, y_mod)

# 3. Prediction shift (Variance) measure karo
shift_opt = np.mean(np.abs(y_opt_grid - opt_mod.predict(poly_opt.transform(X_grid))))
shift_ovr = np.mean(np.abs(y_ovr_grid - ovr_mod.predict(poly_ovr.transform(X_grid))))
print(f"Avg Curve Shift (Variance) -> Normal: {shift_opt:.3f} | Overfit: {shift_ovr:.3f}")

## 🏆 Step 5: Overfitting Key Characteristics Summary

| Property | Overfitting State |
|---|---|
| **Training Accuracy / $R^2$** | Very High (~95%–100%) |
| **Testing Accuracy / $R^2$** | Low (~65%–75% — bada drop) |
| **Model Complexity** | Very High (memorizes outliers) |
| **Bias & Variance** | **Low Bias, High Variance** |

In [ ]:
# 1. Quick verification metric print karo
train_mse_ovr = mean_squared_error(y_train, ovr_model.predict(poly_ovr.transform(X_train)))
test_mse_ovr = mean_squared_error(y_test, ovr_model.predict(poly_ovr.transform(X_test)))

# 2. Cost function J(theta) near zero vs real-world test error compare karo
print(f"Overfit Train MSE (~J_theta): {train_mse_ovr:.4f}")
print(f"Overfit Test MSE (Real Data): {test_mse_ovr:.4f}")